# Research toolkit

Look inside one query-document score: slot attention, the exact slot-pair decomposition, attention-weighted attribution, and causal checks. `breadbowl_embed.research` is **experimental** and may change between minor versions; the core API is stable. Guide: [docs/research.md](../docs/research.md).

Install this repository with the plotting extra first (`python -m pip install ".[viz]"`, or `pip install "breadbowl-embed[viz]"` once it is published on PyPI). The public weights download automatically; set `BREADBOWL_MODEL` to a local export to use that instead. On CPU, `dtype="bfloat16"` is about twice as fast and changes scores by up to about 1e-2.

In [ ]:
import os
from breadbowl_embed import BreadBowl, research

model = BreadBowl.from_pretrained(os.environ.get("BREADBOWL_MODEL", "dotproductx/breadbowl-embed-v1.3"))
query = "How many hearts does an octopus have?"
document = ("Octopuses have three hearts. Two branchial hearts pump blood through the gills, while a single "
            "systemic heart circulates it through the rest of the body. Their blood is blue because it carries "
            "oxygen with copper-based hemocyanin rather than iron-based hemoglobin.")

In [ ]:
pair = research.inspect_pair(model, query, document)
pair  # shaded by attention-weighted attribution of the final score; the slot-pair matrix below

The decomposition recombines to the scorer's own numbers (`model.score`) up to float32 rounding. Each row of the table is one query slot: its share of R and V and the document slot it matches best.

In [ ]:
d = pair.decomposition
print(d.final, d.routing, d.value, d.residuals())
d

Slot attention of the query: how much each slot reads the instruction, the template (`\nQuery:`) and the query text.

In [ ]:
mass = pair.query.segment_mass()
print({segment: [round(float(x), 3) for x in values] for segment, values in mass.items() if segment != "special"})
pair.query

In [ ]:
pair.attribution("document")  # attention-weighted bookkeeping, not causal

## Causal checks

Deleting a sentence re-encodes the passage; `slot-mask` keeps the backbone states and hides a word from the slot extractor only.

In [ ]:
research.occlusion_sweep(model, query, document, unit="sentence", mode="delete")

In [ ]:
sweep = research.occlusion_sweep(model, query, document, unit="word", mode="slot-mask")
sweep.top(5)

In [ ]:
others = ["The Pacific is Earth's largest ocean.", "An octopus has eight arms and a beak."]
queries, documents = model.encode_queries([query]), model.encode_documents([document] + others)
research.compare_rules(queries, documents, base=model)

## Figures and the HTML report

The figures need matplotlib (the `viz` extra). The report is a single file that works offline.

In [ ]:
pair.document.plot(max_tokens=60)
pair.decomposition.plot()
sweep.plot(k=12);

In [ ]:
pair.report("breadbowl-pair-report.html")
pair.save_json("breadbowl-pair.json", digits=6)